In [15]:
import pandas as pd
import json
import math
from rdflib import Graph, Literal, RDF, Namespace, URIRef
from rdflib.namespace import RDFS
from pyvis.network import Network

In [ ]:
#Import and read dataset
df = pd.read_csv("MV_dataset.csv")

#Import and read JSON specification file
with open("feature_spec.json", "r") as file:
    spec = json.load(file)

In [ ]:
#Gather all features from specification file 
feature_map = {}

for symptom in spec["symptoms"]:
    feature_map[symptom["SymptomID"]] = {
        "category": "symptom",
        "fsn": symptom["FSN"],
        "sctid": symptom["SCTID"]
    }

for vital in spec["vitals"]:
    feature_map[vital["VitalSignID"]] = {
        "category": "vital",
        "fsn": vital["FSN"],
        "sctid": vital["SCTID"]
    }

for risk_factor in spec["risk_factors"]:
    feature_map[risk_factor["RiskFactorID"]] = {
        "category": "risk_factor",
        "fsn": risk_factor["FSN"],
        "sctid": risk_factor["SCTID"]
    }

In [36]:
#Initialize empty graph
g = Graph()

#Create namespace for project usign schema.org
SCHEMA = Namespace("http://schema.org/")
SNOMED = Namespace("http://snomed.info/id/")
#Create custom namespace for relationships not covered by standard vocabulary
PAL = Namespace("http://example.org/opioid-toxicity-palliative/")

#Add namespace prefix for nodes
g.bind("schema", SCHEMA)
g.bind("snomed", SNOMED)
g.bind("pal", PAL)

#Iterate through patients
for index, row in df.iterrows():

    #Define patient node
    patient = PAL[str(row["patient_ID"])]
    g.add((patient, RDF.type, SCHEMA.Patient))
    g.add((patient, PAL.hasOpioidToxicityRisk, Literal(row['Outcome'])))

    #Iterate through all symptoms, vitals and risk factors
    for column in feature_map:
        value = str(row[column])
        #Skip features that are not present in patient condition
        if value == "No" or value == "not_asked":
            continue

        #Get feature type, name and snomed code
        category = feature_map[column]["category"]
        name = feature_map[column]["fsn"]
        snomed_id = feature_map[column]["sctid"]
        
        #Define symptom nodes with snomed codes
        if category == "symptom":
            symptom = SNOMED[snomed_id]
            g.add((symptom, RDF.type, SCHEMA.MedicalSymptom))
            g.add((symptom, RDFS.label, Literal(name)))
            g.add((patient, PAL.hasSymptom, symptom))
        
        #Define vital nodes with snomed codes
        elif category == "vital":
            vital = SNOMED[snomed_id]
            g.add((vital, RDF.type, SCHEMA.VitalSign))
            g.add((vital, RDFS.label, Literal(name)))
            g.add((patient,PAL.hasVital, vital))

        #Define risk factor nodes with snomed codes
        elif category == "risk_factor":
            risk_factor = SNOMED[snomed_id]
            g.add((risk_factor, RDF.type, SCHEMA.MedicalRiskFactor))
            g.add((risk_factor, RDFS.label, Literal(name)))
            g.add((patient, PAL.hasRiskFactor, risk_factor))

#Create and print graph
print(g.serialize("knowledge_graph.ttl", format="turtle"))

[a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory']].


In [55]:
import networkx as nx
top_level = {}

for feature_id, feature_info in feature_map.items():
    if feature_id.endswith("-000"):
        top_level[feature_id] = feature_info

G = nx.Graph()

for _, row in df.iterrows():
    pid     = str(row["patient_ID"])
    outcome = str(row["Outcome"])
    G.add_node(pid, label=pid, type="Patient", riskClass=outcome)

    for col in top_level:
        val = str(row[col])
        if val not in ("No", "not_asked"):
            fsn = feature_map[col]["fsn"]
            cat = feature_map[col]["category"]
            G.add_node(fsn, label=fsn, type=cat)
            G.add_edge(pid, fsn, label=f"has_{cat}")

nx.write_graphml(G, "knowledge_graph.graphml")
print("Done -", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

Done - 378 nodes, 3096 edges


In [53]:
from pyvis.network import Network
import math

# ----------------------------------
# Keep only top-level features
# ----------------------------------

top_level = {}

for feature_id, feature_info in feature_map.items():
    if feature_id.endswith("-000"):
        top_level[feature_id] = feature_info


# ----------------------------------
# Create network
# ----------------------------------

net = Network(
    height="900px",
    width="100%",
    bgcolor="white",
    notebook=True
)

# Keep nodes fixed in place
net.toggle_physics(False)


# ----------------------------------
# Define colours
# ----------------------------------

patient_colors = {
    "High": "#e15759",
    "Medium": "#f28e2b",
    "Low": "#59a14f"
}

concept_colors = {
    "symptom": "#4e79a7",
    "vital": "#76b7b2",
    "risk_factor": "#b07aa1"
}


# ----------------------------------
# Find concepts that appear in the data
# ----------------------------------

active_concepts = []

for feature_id in top_level:

    for index, row in df.iterrows():

        value = str(row[feature_id])

        if value != "No" and value != "not_asked":
            active_concepts.append(feature_id)
            break


# ----------------------------------
# Add concept nodes in a circle
# ----------------------------------

for index, feature_id in enumerate(active_concepts):

    feature_info = feature_map[feature_id]

    fsn = feature_info["fsn"]
    sctid = feature_info["sctid"]
    category = feature_info["category"]

    angle = (2 * math.pi * index) / len(active_concepts)

    x_position = 700 * math.cos(angle)
    y_position = 700 * math.sin(angle)

    net.add_node(
        feature_id,
        label=fsn,
        color=concept_colors[category],
        size=25,
        x=x_position,
        y=y_position,
        physics=False,
        title=(
            f"<b>{fsn}</b>"
            f"<br>SNOMED CT: {sctid}"
            f"<br><a href='http://snomed.info/id/{sctid}' target='_blank'>Open in SNOMED CT →</a>"
        )
    )


# ----------------------------------
# Add patient nodes in a grid
# ----------------------------------

grid_columns = 25

for index, row in df.iterrows():

    patient_id = str(row["patient_ID"])
    outcome = row["Outcome"]

    x_position = (index % grid_columns) * 50 - 600
    y_position = (index // grid_columns) * 50 - 100

    net.add_node(
        patient_id,
        label="",
        color=patient_colors[outcome],
        size=8,
        x=x_position,
        y=y_position,
        physics=False,
        title=f"<b>{patient_id}</b><br>Risk Class: {outcome}"
    )


# ----------------------------------
# Add edges
# ----------------------------------

for index, row in df.iterrows():

    patient_id = str(row["patient_ID"])

    for feature_id in top_level:

        value = str(row[feature_id])

        if value != "No" and value != "not_asked":

            category = feature_map[feature_id]["category"]

            net.add_edge(
                patient_id,
                feature_id,
                color={
                    "color": concept_colors[category],
                    "highlight": "#ff6600",
                    "hover": "#ff6600"
                },
                width=0.5
            )


# ----------------------------------
# Save and display graph
# ----------------------------------

net.show("knowledge_graph.html")

knowledge_graph.html


In [ ]:
'''
# Select only core features (top-level, ending in -000)
top_level = {k: v for k, v in feature_map.items() if k.endswith("-000")}

# Define relationships between patient and features
relationship_labels = {
    "symptom":     "has_symptom",
    "vital":       "has_vital",
    "risk_factor": "has_risk_factor"
}

# Define relationship colours for better visibility
edge_colors = {
    "symptom":     "#a8d8ea",
    "vital":       "#b8f0c8",
    "risk_factor": "#f7c6e0"
}

# Sample 5 patients — balanced across risk classes
sample = df.groupby("Outcome", group_keys=False)[df.columns].apply(
    lambda x: x.sample(min(2, len(x)), random_state=42)
).reset_index(drop=True).head(5)

patient_colors = {"High": "#e15759", "Medium": "#f28e2b", "Low": "#59a14f"}
concept_colors = {
    "symptom":     "#4e79a7",
    "vital":       "#76b7b2",
    "risk_factor": "#b07aa1"
}

# Define network graph properties
net = Network(
    height="900px",
    width="100%",
    bgcolor="#ffffff",
    font_color="#333333",
    notebook=True
)
net.toggle_physics(False)

# Patient positions in a grid
grid_cols    = 5
patient_gap  = 150
concept_ring = 400

patient_positions = {}
for i, (_, row) in enumerate(sample.iterrows()):
    px = (i % grid_cols) * patient_gap - (grid_cols - 1) * patient_gap / 2
    py = (i // grid_cols) * patient_gap - patient_gap / 2
    patient_positions[str(row["patient_ID"])] = (px, py)

# Concept node positions — evenly spaced on a large circle
active_concepts = set()
for _, row in sample.iterrows():
    for col in top_level:
        val = str(row[col])
        if val not in ("No", "not_asked"):
            active_concepts.add(col)

active_concepts = list(active_concepts)
concept_positions = {}
for i, col in enumerate(active_concepts):
    angle = (2 * math.pi * i) / len(active_concepts)
    concept_positions[col] = (
        concept_ring * math.cos(angle),
        concept_ring * math.sin(angle)
    )

# Add nodes
added_nodes = set()

def add_concept_node(col):
    if col in added_nodes:
        return
    fsn        = feature_map[col]["fsn"]
    sctid      = feature_map[col]["sctid"]
    cat        = feature_map[col]["category"]
    color      = concept_colors[cat]
    snomed_url = f"http://snomed.info/id/{sctid}"
    cx, cy     = concept_positions[col]
    net.add_node(
        col,
        label=fsn,
        color={"background": color, "border": color},
        size=22,
        x=cx, y=cy,
        physics=False,
        font={"size": 12, "color": "#333333"},
        title=(
            f"<div style='background:white; padding:8px; border:1px solid #ddd;"
            f"border-radius:4px; font-family:Arial; font-size:13px; min-width:180px;'>"
            f"<b>{fsn}</b><br>"
            f"Type: {cat.replace('_', ' ').title()}<br>"
            f"SNOMED CT: {sctid}<br>"
            f"<a href='{snomed_url}' target='_blank' style='color:#4e79a7;'>"
            f"Open in SNOMED CT →</a></div>"
        )
    )
    added_nodes.add(col)

for _, row in sample.iterrows():
    pid     = str(row["patient_ID"])
    outcome = str(row["Outcome"])
    color   = patient_colors[outcome]
    px, py  = patient_positions[pid]

    net.add_node(
        pid,
        label=pid,
        color={"background": color, "border": color},
        size=30,
        x=px, y=py,
        physics=False,
        font={"size": 13, "color": "#333333"},
        title=(
            f"<div style='background:white; padding:8px; border:1px solid #ddd;"
            f"border-radius:4px; font-family:Arial; font-size:13px;'>"
            f"<b>{pid}</b><br>Risk Class: {outcome}</div>"
        )
    )

    for col in top_level:
        val = str(row[col])
        if val in ("No", "not_asked"):
            continue
        cat        = feature_map[col]["category"]
        fsn        = feature_map[col]["fsn"]
        rel        = relationship_labels[cat]
        edge_color = edge_colors[cat]
        add_concept_node(col)

        is_numeric = col in ("OXG-000", "RSP-000", "BPM-000", "AGE-000")
        edge_label = f"{rel}: {val}" if is_numeric else rel

        net.add_edge(
            pid, col,
            label=edge_label,
            color={
                "color":     edge_color,
                "highlight": "#ff6600",
                "hover":     "#ff6600"
            },
            width=2,
            font={
                "size":        11,
                "color":       "#333333",
                "background":  "#ffffff",
                "strokeWidth": 0,
                "align":       "middle"
            },
            title=f"{pid} → {rel} → {fsn}: {val}"
        )

net.set_options("""
{
  "physics": { "enabled": false },
  "interaction": {
    "hover": true,
    "tooltipDelay": 150,
    "navigationButtons": true,
    "hideEdgesOnDrag": false,
    "dragNodes": true,
    "dragView": true,
    "zoomView": true
  },
  "edges": {
    "smooth": { "type": "continuous" },
    "selectionWidth": 4,
    "hoverWidth": 3
  },
  "nodes": {
    "borderWidth": 0,
    "borderWidthSelected": 2
  }
}
""")

net.show("knowledge_graph.html")
'''

knowledge_graph.html
